In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.mongodb.write.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.6.1") \
    .getOrCreate()

df = spark.read.format("mongodb").load()
df.show(5)

+--------------------+-------------------+------------------+--------------------+-----+
|                 _id|      fecha_captura|             grupo|       identificador|valor|
+--------------------+-------------------+------------------+--------------------+-----+
|69d9d4de96a3b9b3e...|2026-04-11 04:57:59|Mercado_Laboral_TI|A Light in the Attic|51.77|
|69d9d4de96a3b9b3e...|2026-04-11 04:57:59|Mercado_Laboral_TI|  Tipping the Velvet|53.74|
|69d9d4de96a3b9b3e...|2026-04-11 04:57:59|Mercado_Laboral_TI|          Soumission| 50.1|
|69d9d4de96a3b9b3e...|2026-04-11 04:57:59|Mercado_Laboral_TI|       Sharp Objects|47.82|
|69d9d4de96a3b9b3e...|2026-04-11 04:57:59|Mercado_Laboral_TI|Sapiens: A Brief ...|54.23|
+--------------------+-------------------+------------------+--------------------+-----+
only showing top 5 rows



In [11]:
from pyspark.sql.functions import col

# Solo queremos registros que tengan un item real y un valor mayor a 0
df_filtrado = df.filter((col("identificador").isNotNull()) & (col("valor") > 0))

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros validos:", df_filtrado.count())

PASO 2: Limpieza completada.
Registros originales: 60
Registros validos: 60


In [12]:
df.select("identificador", "valor").show()

+--------------------+-----+
|       identificador|valor|
+--------------------+-----+
|A Light in the Attic|51.77|
|  Tipping the Velvet|53.74|
|          Soumission| 50.1|
|       Sharp Objects|47.82|
|Sapiens: A Brief ...|54.23|
|     The Requiem Red|22.65|
|The Dirty Little ...|33.34|
|The Coming Woman:...|17.93|
|The Boys in the B...| 22.6|
|     The Black Maria|52.15|
|Starving Hearts (...|13.99|
|Shakespeare's Son...|20.66|
|         Set Me Free|17.46|
|Scott Pilgrim's P...|52.29|
|Rip it Up and Sta...|35.02|
|Our Band Could Be...|57.25|
|                Olio|23.88|
|Mesaerion: The Be...|37.59|
|Libertarianism fo...|51.33|
|It's Only the Him...|45.17|
+--------------------+-----+
only showing top 20 rows



In [13]:
df.filter(df["valor"] > 50).show()

+--------------------+-------------------+------------------+--------------------+-----+
|                 _id|      fecha_captura|             grupo|       identificador|valor|
+--------------------+-------------------+------------------+--------------------+-----+
|69d9d4de96a3b9b3e...|2026-04-11 04:57:59|Mercado_Laboral_TI|A Light in the Attic|51.77|
|69d9d4de96a3b9b3e...|2026-04-11 04:57:59|Mercado_Laboral_TI|  Tipping the Velvet|53.74|
|69d9d4de96a3b9b3e...|2026-04-11 04:57:59|Mercado_Laboral_TI|          Soumission| 50.1|
|69d9d4de96a3b9b3e...|2026-04-11 04:57:59|Mercado_Laboral_TI|Sapiens: A Brief ...|54.23|
|69d9d4de96a3b9b3e...|2026-04-11 04:58:00|Mercado_Laboral_TI|     The Black Maria|52.15|
|69d9d4de96a3b9b3e...|2026-04-11 04:58:00|Mercado_Laboral_TI|Scott Pilgrim's P...|52.29|
|69d9d4de96a3b9b3e...|2026-04-11 04:58:00|Mercado_Laboral_TI|Our Band Could Be...|57.25|
|69d9d4de96a3b9b3e...|2026-04-11 04:58:00|Mercado_Laboral_TI|Libertarianism fo...|51.33|
|69d9d4de96a3b9b3e...

In [14]:
df.groupBy("grupo").count().show()

+------------------+-----+
|             grupo|count|
+------------------+-----+
|Mercado_Laboral_TI|   60|
+------------------+-----+



In [16]:
# 2. Transformacion y Agregacion por CATEGORIA
# Esto permite comparar que generos son mas caros en promedio
reporte_categorias = df.groupBy("grupo").agg(
    F.count("identificador").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR CATEGORIA:")
reporte_categorias.show()

ANALISIS DE MERCADO POR CATEGORIA:
+------------------+---------------+-----------------+-------------+-------------+
|             grupo|cantidad_libros|  precio_promedio|precio_minimo|precio_maximo|
+------------------+---------------+-----------------+-------------+-------------+
|Mercado_Laboral_TI|             60|35.00266666666666|        12.84|        57.31|
+------------------+---------------+-----------------+-------------+-------------+



In [19]:
fila = df.orderBy(F.desc("valor")).select(
    "identificador", "grupo", "fecha_captura", "valor"
).first()

print("Producto más caro:", fila["identificador"])
print("Grupo:", fila["grupo"])
print("Fecha y hora de captura:", fila["fecha_captura"])
print("Valor:", fila["valor"])

Producto más caro: Slow States of Collapse: Poems
Grupo: Mercado_Laboral_TI
Fecha y hora de captura: 2026-04-11 04:58:05
Valor: 57.31
